# Smoke A/B — three attacks × live agents (GGUF)

**Attacks:** `ours_no_harmony` / `ours_harmony` / `tetsutani`

**Not a Submit.** Short budget. GPU ON, Internet OFF.

## Important Kaggle quirks
- **Idle restart:** leaving the editor tab often kills the session. Prefer **Save Version → Save & Run All** (background run) OR keep tab open/focused.
- **Silent load:** first `Llama(...)` GGUF load can take 10–20+ min with little cell output. Watch `/kaggle/working/smoke_heartbeat.txt` in the right Output panel (refresh).
- Default: **gpt_oss only** (faster). Uncomment gemma later.

## Setup
1. Attach competition + both GGUF models
2. GPU on, Internet off
3. Run All — or Save Version for durable run


In [ ]:
import sys, os, glob, time
from pathlib import Path

sys.argv = [sys.argv[0]]

def log(msg: str) -> None:
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line, flush=True)
    try:
        p = Path('/kaggle/working/smoke_heartbeat.txt')
        p.parent.mkdir(parents=True, exist_ok=True)
        with p.open('a', encoding='utf-8') as f:
            f.write(line + '\n')
    except Exception:
        pass

log('cell1: start path probe')
dataset_root = None
for candidate in glob.glob('/kaggle/input/**/kaggle_evaluation', recursive=True):
    dataset_root = str(Path(candidate).parent)
    if dataset_root not in sys.path:
        sys.path.insert(0, dataset_root)
    break
log(f'dataset_root={dataset_root}')
ggufs = sorted(glob.glob('/kaggle/input/**/*.gguf', recursive=True))
log(f'gguf_count={len(ggufs)}')
for p in ggufs[:30]:
    log(f'  {p} ({Path(p).stat().st_size/1e9:.2f} GB)')
assert dataset_root, 'Attach competition input that contains kaggle_evaluation/'


In [ ]:
ATTACKS_B64 = {
    "ours_no_harmony": (
        "..."
        "Y2FuZGlkYXRlc1s6TUFYX0NBTkRJREFURVNdCg=="
    ),
    "ours_harmony": (
        "..."
    ),
    "tetsutani": (
        "..."
        "RVNdCg=="
    ),
}

from pathlib import Path
import base64, time

def log(msg: str) -> None:
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line, flush=True)
    try:
        with Path('/kaggle/working/smoke_heartbeat.txt').open('a', encoding='utf-8') as f:
            f.write(line + '\n')
    except Exception:
        pass

ATTACK_DIR = Path('/kaggle/working/ab_attacks')
ATTACK_DIR.mkdir(parents=True, exist_ok=True)
for name, b64 in ATTACKS_B64.items():
    blob = b64 if isinstance(b64, str) else ''.join(b64)
    path = ATTACK_DIR / f'{name}.py'
    path.write_bytes(base64.b64decode(blob))
    log(f'wrote {path.name} ({path.stat().st_size} bytes)')


In [ ]:
import glob, os, time
from pathlib import Path

def log(msg: str) -> None:
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line, flush=True)
    try:
        with Path('/kaggle/working/smoke_heartbeat.txt').open('a', encoding='utf-8') as f:
            f.write(line + '\n')
    except Exception:
        pass

def pick_gguf(*needles: str) -> str | None:
    hits = []
    for p in glob.glob('/kaggle/input/**/*.gguf', recursive=True):
        low = p.lower()
        if all(n.lower() in low for n in needles):
            hits.append(p)
    if not hits and needles:
        for p in glob.glob('/kaggle/input/**/*.gguf', recursive=True):
            if needles[0].lower() in p.lower():
                hits.append(p)
    if not hits:
        return None
    hits.sort(key=lambda p: (Path(p).stat().st_size, p))
    return hits[0]

GPT_OSS_GGUF = pick_gguf('gpt-oss') or pick_gguf('gpt_oss') or pick_gguf('oss-20')
GEMMA_GGUF = pick_gguf('gemma-4') or pick_gguf('gemma4') or pick_gguf('gemma')

log(f'GPT_OSS_GGUF={GPT_OSS_GGUF}')
log(f'GEMMA_GGUF={GEMMA_GGUF}')

BUDGET_S = 60.0       # short smoke; raise later
MAX_TOOL_HOPS = 8
N_CTX = 4096          # lower VRAM / faster load than 8192
N_GPU_LAYERS = -1

# Default: gpt_oss only (avoid 2x long loads + idle kill)
AGENTS = []
if GPT_OSS_GGUF:
    AGENTS.append(('gpt_oss', GPT_OSS_GGUF))
# Uncomment for gemma after gpt_oss smoke works:
# if GEMMA_GGUF:
#     AGENTS.append(('gemma_4', GEMMA_GGUF))

log(f'AGENTS={AGENTS}')
log(f'BUDGET_S={BUDGET_S} N_CTX={N_CTX}')
assert AGENTS, 'No GGUF under /kaggle/input — attach models first'


In [ ]:
import importlib.util
import time
from pathlib import Path

import aicomp_sdk
from aicomp_sdk import AttackRunConfig
from aicomp_sdk.agents.gpt_oss_agent import GPTOSSAgent, build_gpt_oss_backend_config, build_gpt_oss_parser
from aicomp_sdk.agents.gemma4_agent import Gemma4Agent, build_gemma4_backend_config, build_gemma4_parser
from aicomp_sdk.agents.hf_chat_template.backends.llama_cpp import LlamaCppChatTemplateBackend
from aicomp_sdk.core.env.sandbox import SandboxEnv
from aicomp_sdk.guardrails.optimal import Guardrail as OptimalGuardrail

ATTACK_DIR = Path('/kaggle/working/ab_attacks')
ATTACK_NAMES = ['ours_no_harmony', 'ours_harmony', 'tetsutani']


def log(msg: str) -> None:
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line, flush=True)
    try:
        with Path('/kaggle/working/smoke_heartbeat.txt').open('a', encoding='utf-8') as f:
            f.write(line + '\n')
    except Exception:
        pass


def load_attack(name: str):
    path = ATTACK_DIR / f'{name}.py'
    spec = importlib.util.spec_from_file_location(f'ab_{name}', path)
    mod = importlib.util.module_from_spec(spec)
    import sys
    sys.modules[spec.name] = mod
    spec.loader.exec_module(mod)
    return mod


def make_gguf_agent(agent_name: str, gguf_path: str):
    log(f'loading llama_cpp backend for {agent_name} from {gguf_path}')
    t0 = time.time()
    if agent_name == 'gpt_oss':
        cfg = build_gpt_oss_backend_config(model_path=gguf_path, local_files_only=True)
        backend = LlamaCppChatTemplateBackend.from_model_path(
            model_path=gguf_path,
            config=cfg,
            n_ctx=N_CTX,
            n_gpu_layers=N_GPU_LAYERS,
            verbose=False,
        )
        agent = GPTOSSAgent(
            backend,
            parser=build_gpt_oss_parser(None, model_id=cfg.model_id or 'openai/gpt-oss-20b'),
        )
    elif agent_name == 'gemma_4':
        cfg = build_gemma4_backend_config(model_path=gguf_path, local_files_only=True)
        backend = LlamaCppChatTemplateBackend.from_model_path(
            model_path=gguf_path,
            config=cfg,
            n_ctx=N_CTX,
            n_gpu_layers=N_GPU_LAYERS,
            verbose=False,
        )
        agent = Gemma4Agent(
            backend,
            parser=build_gemma4_parser(None, model_id=cfg.model_id or 'google/gemma-4'),
        )
    else:
        raise ValueError(agent_name)
    log(f'loaded {agent_name} in {time.time()-t0:.1f}s')
    return agent


def make_env(agent_name: str, gguf_path: str):
    fixtures = Path(aicomp_sdk.__file__).resolve().parent / 'fixtures'
    return SandboxEnv(
        seed=1,
        fixtures_dir=fixtures,
        agent=make_gguf_agent(agent_name, gguf_path),
        guardrail=OptimalGuardrail(),
    )


results = []
log('smoke loop start')
for agent_name, gguf_path in AGENTS:
    log('=' * 60)
    log(f'AGENT {agent_name}')
    env = make_env(agent_name, gguf_path)
    for attack_name in ATTACK_NAMES:
        log(f'--- {attack_name} @ {agent_name} budget={BUDGET_S}s ---')
        mod = load_attack(attack_name)
        summary = Path(f'/kaggle/working/summary_{agent_name}_{attack_name}.txt')
        if hasattr(mod, 'SUMMARY_PATHS'):
            mod.SUMMARY_PATHS = (summary, Path(f'summary_{agent_name}_{attack_name}.txt'))
        t0 = time.time()
        try:
            findings = mod.AttackAlgorithm().run(
                env,
                AttackRunConfig(time_budget_s=BUDGET_S, max_tool_hops=MAX_TOOL_HOPS),
            )
            err = ''
        except Exception as e:
            findings = []
            err = f'{type(e).__name__}: {e}'
            log(f'ERROR {err}')
        elapsed = time.time() - t0
        n = len(findings) if findings is not None else 0
        notes = ''
        if summary.exists():
            text = summary.read_text(encoding='utf-8')
            log('summary:\n' + text[:800])
            for line in text.splitlines():
                if line.startswith('selected=') or line.startswith('core_best='):
                    notes += line + ' '
        results.append({
            'agent': agent_name,
            'attack': attack_name,
            'findings': n,
            'wall_s': round(elapsed, 1),
            'error': err,
            'notes': notes.strip(),
            'summary_path': str(summary) if summary.exists() else '',
        })
        log(f'done findings={n} wall_s={elapsed:.1f}')

log('DONE smoke loop')


In [ ]:
import pandas as pd
import time
from pathlib import Path

def log(msg: str) -> None:
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line, flush=True)
    try:
        with Path('/kaggle/working/smoke_heartbeat.txt').open('a', encoding='utf-8') as f:
            f.write(line + '\n')
    except Exception:
        pass

df = pd.DataFrame(results)
log('\n' + df.to_string(index=False))
out = Path('/kaggle/working/ab_smoke_results.csv')
df.to_csv(out, index=False)
log(f'wrote {out}')

for agent, g in df.groupby('agent'):
    ok = g[g['error'] == '']
    if ok.empty:
        log(f'{agent} all failed')
        continue
    best = ok.sort_values(['findings', 'wall_s'], ascending=[False, True]).iloc[0]
    log(
        f"{agent} best: {best['attack']} findings={best['findings']} wall_s={best['wall_s']} {best.get('notes','')}"
    )
